# Exploracion con NLP y LLMs

ideas generales

- Cómo se presenta y se nombra a las personas que aparecen en el vídeo: nombre, título, cargo, profesión (cruzado con el OCR). ¿Se nombra con título y cargo por igual a mujeres y hombres? ("la Doctora María" vs. "mi compañera María")
	- Task 1: LLM. Extraer cada referencia a una persona presente: cómo se la nombra, nombre, título/cargo y género.

- Temas según el género de quien habla: estereotípicos (cuidados, familia, vida sentimental, ser mujer) vs. profesionales (tecnología, finanzas, liderazgo, su trabajo).
	- Task 2: Zero-shot o LLM con una lista fija de temas, aplicado a cada turno. Con el género del hablante (del audio), sale el reparto de temas por género.
    - Aquí a lo mejor tengo que linkear a qién se refiere el topic, no solo quién habla de él

- Referentes: ¿se citan personas destacadas femeninas o solo masculinas?
	- Task 3: NER con GLiNER para encontrar los nombres y buscarlos en Wikidata para saber su género.

- Rol de cada persona en la pieza. Primero el formato (entrevista, reportaje con voz en off, narración, anuncio, debate…) y después el rol de cada hablante según el formato: experta/o, entrevistador/a, presentador/a, narrador/a en off, testimonio, personaje… ¿Quién tiene los roles de autoridad?
	- Task 4: LLM con la transcripción completa: formato de la pieza y rol de cada hablante.

- Desprecios, desmerecimiento, micromachismos o faltas de respeto, y su dirección (mujer → hombre, hombre → mujer). Las interrupciones se miden en el pipeline de audio.
	- Task 5: LLM con criterios escritos y revisión humana o probar modelos zero shot entrenados para esta tarea.

- Contribución al contenido: si quitamos todas las intervenciones femeninas, ¿sigue teniendo sentido la pieza?
	- Task 6: LLM resume la pieza completa: % de las ideas del resumen que vienen de intervenciones de mujeres. Complemento: LLM como juez con una rúbrica de escala cerrada.


- Task 7: Cuantificar con word2vec traduciendo al inglés, medir con sesgo de estereotipo
- Validación: anotar a mano 2–3 vídeos para comprobar cada task.

## 0. Configuración

Entrada: la transcripción del pipeline de audio (CSV del contrato: `tiempo_i`, `tiempo_f`, `transcripcion`, `genero_inferido`).

In [1]:
import os
from typing import Literal
import pandas as pd
import litellm
from pydantic import BaseModel, Field

TRANSCRIPT = "clip_quanta.csv"            # <- CSV de la transcripción

BASE = os.environ.get("OVH_BASE_URL", "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1")
TOKEN = "".join((os.environ.get("OVH_AI_ENDPOINTS_ACCESS_TOKEN") or os.environ.get("OVH_API_TOKEN") or "").split())
MODEL = "openai/Meta-Llama-3_3-70B-Instruct"   # "openai/" = endpoint compatible con OpenAI (OVH)
assert TOKEN, "Falta el token: define la variable de entorno OVH_AI_ENDPOINTS_ACCESS_TOKEN"

turnos = pd.read_csv(TRANSCRIPT)
# La transcripción en texto: un turno por línea, con su minuto y segundo
texto = "\n".join(f"[{int(t // 60):02d}:{int(t % 60):02d}] {txt}" for t, txt in zip(turnos.tiempo_i, turnos.transcripcion))
print(texto[:500])

[00:00] declarada sobre aquesta qüestió. Ella és la Carina Givert, és la directora i cofundadora de l'IDEA i la coneixeu, Carina. Benvinguda. Hola. Et puc fer la presentació oficial? Sí, és encantada. Presentació oficial.
[00:15] Carina Givert és catedràtica a la Universitat Politécnica de Catalunya Barcelona Tech. Llicenciada i doctora en informàtica amb especialitzacions en estadística computacional i intel·ligència artificial. directora i cofundadora de l'IDEAI Centre de Recerca de la UPC en 


## Task 1. Cómo se nombra a las personas de la conversación

Cada alusión a una persona que participa en la pieza (presentaciones, cuando alguien se dirige a ella, referencias): texto literal, nombre, título o cargo y género. Salida estructurada con Pydantic; el prompt incluye ejemplos (*few-shot*).

In [6]:
class Alusion(BaseModel):
    texto_literal: str = Field(description="Fragmento COMPLETO literal de la transcripción donde se alude a la persona")
    nombre: str | None = Field(description="Nombre propio de la persona; null si no se dice")
    titulo_cargo: str | None = Field(description="Título, cargo o profesión; null si no se dice")
    genero: Literal["F", "M"] | None = Field(description="Género gramatical con el que se alude a la persona; null si no se sabe, o no es binario.")

class Alusiones(BaseModel):
    alusiones: list[Alusion]

In [7]:
PROMPT_T1 = """Analizas la transcripción automática de un vídeo (puede estar en español, catalán o inglés y tener errores de reconocimiento de voz).
Extrae TODAS las alusiones a personas que participan en la conversación: cuando se las presenta, cuando alguien se dirige a ellas o cuando se habla de ellas.
No incluyas a personas que solo se citan y no participan explícitamente (personajes históricos, famosos) NI PRONOMBRES.

Para cada alusión devuelve:
- texto_literal: el fragmento completo exacto de la transcripción.
- nombre: el nombre propio tal como aparece; null si no se dice.
- titulo_cargo: título, cargo o profesión (doctora, catedrática de la UPC, director del hospital...); null si no se dice.
  Las relaciones NO son cargos: "mi compañera", "mi amigo", "la mujer de..." -> null.
- genero: "F" o "M" según cómo se alude a la persona (artículos, pronombres, concordancia); null si no se sabe.

Ejemplos:
Texto: "Hoy nos acompaña la doctora Marta Ruiz, investigadora del CSIC."
-> {"texto_literal": "la doctora Marta Ruiz, investigadora del CSIC", "nombre": "Marta Ruiz", "titulo_cargo": "doctora, investigadora del CSIC", "genero": "F"}
Texto: "Gracias, Luis, por estar aquí."
-> {"texto_literal": "Gracias, Luis", "nombre": "Luis", "titulo_cargo": null, "genero": "M"}
Texto: "Mi compañera nos lo explica ahora."
-> {"texto_literal": "Mi compañera", "nombre": null, "titulo_cargo": null, "genero": "F"}
Texto: "Pere Vila, director del hospital, ha atendido a los medios."
-> {"texto_literal": "Pere Vila, director del hospital", "nombre": "Pere Vila", "titulo_cargo": "director del hospital", "genero": "M"}"""

resp = litellm.completion(
    model=MODEL, api_base=BASE, api_key=TOKEN, temperature=0,
    messages=[{"role": "system", "content": PROMPT_T1},
              {"role": "user", "content": texto}],
    response_format=Alusiones,             
)
alusiones = Alusiones.model_validate_json(resp.choices[0].message.content)
t1 = pd.DataFrame([a.model_dump() for a in alusiones.alusiones])
t1

,texto_literal,nombre,titulo_cargo,genero
0,"Ella és la Carina Givert, és la directora i co...",Carina Givert,directora i cofundadora de l'IDEA,F
1,Carina Givert és catedràtica a la Universitat ...,Carina Givert,catedràtica a la Universitat Politècnica de Ca...,F
2,directora i cofundadora de l'IDEAI Centre de R...,Carina Givert,directora i cofundadora de l'IDEAI Centre de R...,F
3,"Carina, què ens pots explicar d'aquestes top s...",Carina,NaN,F
4,aquestes dones,NaN,NaN,F
5,dones com elles,NaN,NaN,F
6,"Carina, què ens pots explicar d'aquestes top s...",Rosi,NaN,F


## Task 2. Temas según el género de quien habla

Juntamos por separado lo que dicen las mujeres y lo que dicen los hombres (`genero_inferido` del audio; los turnos `no_determinado` se excluyen) y pasamos el LLM una vez por cada texto. `TEMAS_GUIA` es solo una guía: el modelo puede proponer temas nuevos.

In [8]:
# Guía de temas (editable): el modelo la usa como referencia, pero puede añadir temas nuevos
TEMAS_GUIA = [
    # estereotipados como "femeninos" o centrados en la persona
    "cuidados", "familia y maternidad", "conciliación", "vida sentimental y relaciones",
    "aspecto físico", "ser mujer / experiencia de género", "diversidad e igualdad",
    # profesionales o estereotipados como "masculinos"
    "tecnología", "ciencia e investigación", "finanzas y economía", "liderazgo y gestión",
    "su trabajo o especialidad", "política", "deporte",
    # otros
    "educación", "salud", "historia",
]

In [9]:
class Tema(BaseModel):
    tema: str = Field(description="Nombre corto del tema: uno de la guía si encaja, o uno nuevo")
    evidencia: str = Field(description="Fragmento literal breve de la transcripción que muestra el tema")

class Temas(BaseModel):
    temas: list[Tema]

PROMPT_T2 = f"""Analizas lo que dicen algunas personas en un vídeo (transcripción automática; puede estar en español, catalán o inglés y tener errores).
Enumera los temas de los que hablan. Para cada tema da una evidencia: un fragmento literal breve del texto.
Usa como guía esta lista de temas y, si uno encaja, escríbelo exactamente igual: {", ".join(TEMAS_GUIA)}.
Si se habla de algo que no está en la lista, crea un tema nuevo con un nombre corto en español."""

def extraer_temas(texto):
    resp = litellm.completion(
        model=MODEL, api_base=BASE, api_key=TOKEN, temperature=0,
        messages=[{"role": "system", "content": PROMPT_T2},
                  {"role": "user", "content": texto}],
        response_format=Temas,
    )
    return Temas.model_validate_json(resp.choices[0].message.content).temas

filas = []
for genero in ["F", "M"]:
    texto_genero = "\n".join(turnos[turnos.genero_inferido == genero].transcripcion)
    if not texto_genero:                     # en la pieza no habla nadie de ese género
        print(f"Sin turnos de género {genero}")
        continue
    filas += [{"genero": genero, **t.model_dump()} for t in extraer_temas(texto_genero)]

t2 = pd.DataFrame(filas)
t2["en_guia"] = t2.tema.str.lower().isin([t.lower() for t in TEMAS_GUIA])
t2

Sin turnos de género M


,genero,tema,evidencia,en_guia
0,F,diversidad e igualdad,dones com elles i unes quantes més que han est...,True
1,F,tecnología,"la informàtica la coneixem avui, és gràcies en...",True
2,F,ciencia e investigación,es van inventar el concepte de rutina informàt...,True
3,F,liderazgo y gestión,"la Carina Givert, és la directora i cofundador...",True


In [ ]:
# Qué temas trata cada género (1 = aparece)
pd.crosstab(t2.tema, t2.genero)

## Task 3. Referentes: ¿se mencionan mujeres famosas?

1. [GLiNER2.5 multi](https://github.com/fastino-ai/GLiNER2) busca nombres de personas famosas en cada turno (quitamos a quien participa, de la Task 1).
2. El LLM da el género de cada nombre (`null` si no es una persona).
3. [`yangheng/deberta-v3-base-absa-v1.1`](https://huggingface.co/yangheng/deberta-v3-base-absa-v1.1) da el sentimiento con el que se habla de cada persona en su frase.

In [ ]:
from gliner2 import AutoExtractor

gliner = AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1")     # la primera vez descarga ~1 GB
MIN_CONF = 0.7

filas = []
for _, turno in turnos.iterrows():
    res = gliner.extract_entities(turno.transcripcion, ["persona famosa"], include_confidence=True)
    filas += [{"tiempo_i": turno.tiempo_i, "nombre": p["text"], "frase": turno.transcripcion}
              for p in res["entities"]["persona famosa"] if p["confidence"] >= MIN_CONF]
referentes = pd.DataFrame(filas, columns=["tiempo_i", "nombre", "frase"])

# Quitamos a quien participa (Task 1)
participantes = " ".join(t1.nombre.dropna()).lower()
referentes = referentes[~referentes.nombre.str.lower().apply(lambda n: n in participantes)]
referentes

In [ ]:
class GeneroPersona(BaseModel):
    nombre: str
    genero: Literal["F", "M"] | None = Field(description="null si no se sabe o no es una persona")

class Generos(BaseModel):
    personas: list[GeneroPersona]

PROMPT_T3 = "Indica el género (F o M) de cada persona de la lista; null si no se sabe o si no es una persona. Devuelve el nombre tal cual."

resp = litellm.completion(
    model=MODEL, api_base=BASE, api_key=TOKEN, temperature=0,
    messages=[{"role": "system", "content": PROMPT_T3},
              {"role": "user", "content": "\n".join(referentes.nombre.unique())}],
    response_format=Generos,
)
generos = {p.nombre: p.genero for p in Generos.model_validate_json(resp.choices[0].message.content).personas}
referentes["genero"] = referentes.nombre.map(generos)

In [ ]:
from transformers import pipeline

# Sentimiento hacia la persona en su frase (no de la frase entera): se le pasa la frase y el nombre
absa = pipeline("text-classification", model="yangheng/deberta-v3-base-absa-v1.1")
referentes["sentimiento"] = [absa(f, text_pair=n)[0]["label"] for f, n in zip(referentes.frase, referentes.nombre)]
referentes.drop(columns="frase")

In [ ]:
# KPI: % de referentes únicos por género y sentimiento por género
print((100 * referentes.drop_duplicates("nombre").genero.value_counts(normalize=True)).round(1))
pd.crosstab(referentes.genero, referentes.sentimiento)

### Task 3.1 Evaluación: ¿qué enfoque usar?

Texto de prueba: el podcast *Quanta* de betevé sobre mujeres en la IA. Referencia anotada a mano: **12 personas famosas mencionadas (7 mujeres, 5 hombres)**, varias mal transcritas ("Heidi Lamar", "Guedel") o citadas solo por el apellido ("Turing", "Llull").

Para cada enfoque medimos cuántas encuentra, a cuántas les asigna el género correcto y cuántos **falsos positivos** extrae (personas no famosas o que no existen).

*DBpedia Spotlight* quedó fuera: su API pública no responde y la imagen oficial de Docker no consigue descargar el modelo catalán (comprobado el 22/10/2026).

In [ ]:
import json, re, unicodedata
import requests

# Texto de prueba: podcast Quanta (betevé) sobre mujeres en la IA, con referentes de los dos géneros
EVAL_JSON = "la-ia-i-el-talent-femeni-ytmp4.savetube.vip.json"
eval_turnos = [t["text"] for t in json.load(open(EVAL_JSON))["diarization"]]
eval_texto = " ".join(eval_turnos)

# Referencia anotada a mano: persona famosa -> (género, cómo aparece en la transcripción)
GOLD = {
    "Margaret Hamilton": ("F", ["Margaret Hamilton"]),
    "Katherine Johnson": ("F", ["Katherine Johnson"]),
    "Hedy Lamarr":       ("F", ["Heidi Lamar"]),
    "Ada Lovelace":      ("F", ["Ada Lovelace"]),
    "Rózsa Péter":       ("F", ["Rosa Peter"]),
    "Angela Merkel":     ("F", ["Angela Merkel"]),
    "Isabel II":         ("F", ["reina d'Inglaterra"]),
    "Aristóteles":       ("M", ["Aristoti"]),
    "Ramon Llull":       ("M", ["Llull"]),
    "Kurt Gödel":        ("M", ["Guedel"]),
    "Alan Turing":       ("M", ["Turing"]),
    "Francisco Franco":  ("M", ["Franco"]),
}
# No cuentan ni a favor ni en contra: participantes, grupos y nombres dudosos
IGNORAR = ["carina", "givert", "sara peter", "top secret", "top roses", "rosi", "icatia"]

def norm(s):
    """Minúsculas y sin tildes, para comparar nombres."""
    return unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()

def evaluar(enfoque, predicciones):
    """predicciones: lista de (nombre, género). Devuelve aciertos, género correcto y falsos positivos."""
    encontradas, falsos = {}, set()
    for nombre, genero in predicciones:
        n = norm(nombre)
        persona = next((p for p, (_, formas) in GOLD.items() if any(norm(f) in n for f in formas)), None)
        if persona:
            encontradas[persona] = genero
        elif not any(i in n for i in IGNORAR):
            falsos.add(nombre)
    genero_ok = [p for p, g in encontradas.items() if g == GOLD[p][0]]
    return {"enfoque": enfoque,
            "encontradas": f"{len(encontradas)}/{len(GOLD)}",
            "género correcto": f"{len(genero_ok)}/{len(GOLD)}",
            "falsos positivos": len(falsos),
            "no encontradas": ", ".join(p for p in GOLD if p not in encontradas),
            "género mal": ", ".join(p for p in encontradas if p not in genero_ok),
            "ejemplos de falsos positivos": ", ".join(sorted(falsos)[:6])}

# Género en Wikidata (P21) de una entidad, solo si es una persona (P31 = Q5)
WIKIDATA = "https://www.wikidata.org/w/api.php"
# Wikimedia exige un User-Agent con contacto; sin él, responde 429 (demasiadas peticiones)
HEADERS = {"User-Agent": "attention-is-all-she-needs/0.1 (https://github.com/ComunidadIA-OS/attention-is-all-she-needs)"}
SEXO = {"Q6581072": "F", "Q6581097": "M"}

def genero_si_persona(claims):
    """None si la entidad no es una persona; si lo es, su género (F/M o None si no consta)."""
    if not any(c["mainsnak"].get("datavalue", {}).get("value", {}).get("id") == "Q5" for c in claims.get("P31", [])):
        return None
    sexo = [c["mainsnak"]["datavalue"]["value"]["id"] for c in claims.get("P21", []) if "datavalue" in c["mainsnak"]]
    return SEXO.get(sexo[0], "otro") if sexo else "desconocido"

In [ ]:
# Enfoque 1: solo GLiNER2, que también da el género ("famous woman" / "famous man"; gana la etiqueta de más confianza)
from gliner2 import AutoExtractor
gliner = AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1")
frases = [f for t in eval_turnos for f in re.split(r"(?<=[.!?])\s+", t) if f.strip()]   # GLiNER2 solo lee el principio de textos largos

ETIQUETAS = {"famous woman": "F", "famous man": "M"}
mejor = {}                                            # nombre -> (confianza, género)
for f in frases:
    res = gliner.extract_entities(f, list(ETIQUETAS), include_confidence=True)["entities"]
    for etiqueta, genero in ETIQUETAS.items():
        for p in res.get(etiqueta, []):
            if p["confidence"] >= 0.7 and p["text"][0].isupper() and p["confidence"] > mejor.get(p["text"], (0, None))[0]:
                mejor[p["text"]] = (p["confidence"], genero)
res_1 = evaluar("1. GLiNER2 solo", [(n, g) for n, (_, g) in mejor.items()])
res_1

In [ ]:
# Enfoque 2: GLiNER2 detecta "persona famosa" + Wikidata da el género del primer resultado que sea una persona
import time

def wikidata(params):
    time.sleep(0.5)                                   # sin prisa: respetamos los límites de Wikimedia
    r = requests.get(WIKIDATA, headers=HEADERS, params={**params, "format": "json"})
    r.raise_for_status()
    return r.json()

def wikidata_primera_persona(nombre):
    ids = [r["id"] for r in wikidata({"action": "wbsearchentities", "search": nombre, "language": "ca", "limit": 5})["search"]]
    if not ids:
        return None
    entidades = wikidata({"action": "wbgetentities", "ids": "|".join(ids), "props": "claims"})["entities"]
    return next((g for i in ids if (g := genero_si_persona(entidades[i]["claims"]))), None)

nombres = {p["text"] for f in frases
           for p in gliner.extract_entities(f, ["persona famosa"], include_confidence=True)["entities"]["persona famosa"]
           if p["confidence"] >= 0.7 and p["text"][0].isupper()}
# Si Wikidata no tiene ninguna persona con ese nombre, no la consideramos famosa
res_2 = evaluar("2. GLiNER2 + Wikidata", [(n, g) for n in nombres if (g := wikidata_primera_persona(n))])
res_2

In [ ]:
# Enfoque 3: el LLM detecta las personas famosas y su género en una sola llamada
class Referente(BaseModel):
    nombre: str = Field(description="Nombre completo de la persona, corregido si la transcripción lo escribe mal")
    texto_literal: str = Field(description="Cómo aparece EXACTAMENTE en la transcripción")
    genero: Literal["F", "M"] | None

class Referentes(BaseModel):
    referentes: list[Referente]

PROMPT_REF = """Extrae las personas famosas o históricas que se mencionan en esta transcripción automática (puede tener errores en los nombres).
No incluyas a quienes participan en la conversación ni a personas anónimas o genéricas ("una monja", "mi madre"...).
Para cada una: nombre (corregido si está mal transcrito), texto_literal (cómo aparece exactamente en el texto) y genero (F o M)."""

resp = litellm.completion(
    model=MODEL, api_base=BASE, api_key=TOKEN, temperature=0,
    messages=[{"role": "system", "content": PROMPT_REF}, {"role": "user", "content": eval_texto}],
    response_format=Referentes,
)
refs = Referentes.model_validate_json(resp.choices[0].message.content).referentes
# Control anti-invención: solo cuenta si el texto literal aparece de verdad en la transcripción
res_3 = evaluar("3. LLM", [(r.texto_literal, r.genero) for r in refs if norm(r.texto_literal) in norm(eval_texto)])
res_3

In [ ]:
pd.DataFrame([res_1, res_2, res_3]).set_index("enfoque")

**Resultados sin el LLM** (22/10/2026):
- **GLiNER2 solo**: encuentra 10/12, pero acierta el género solo en 7/12. Los hombres citados por el apellido en una conversación sobre mujeres (Turing, Llull, Gödel) salen como mujeres, lo que infla justo el KPI que queremos medir.
- **GLiNER2 + Wikidata**: 9/12 con el género correcto y 0 falsos positivos, pero pierde los nombres mal transcritos (no los encuentra en Wikidata) y depende de una API externa con límites de uso.

Ejecutad la celda del LLM para completar la tabla.